# Data Retrieval Polls

| Field | Details |
|---|---|
| **Time window** | Jul 2024 – Nov 2024 |
| **Source** | Wikipedia — *Nationwide opinion polling for the 2024 United States presidential election* |
| **Method** | Web scraping with `BeautifulSoup` — parsing HTML tables |
| **Content** | Pollster · date · sample size · Trump % · Harris % · margin |
| **Volume** | **256** polls from ~40 different polling firms |
| **Saved to** | `Data/1_Bronze/Polls/wikipedia_polls.csv` |

### wikipedia_polls.csv 256 rows × 6 columns

| Column | Type | Description |
|---|---|---|
| `Pollster` | str | Name of the polling organisation |
| `Date` | date | Poll fieldwork end date |
| `SampleSize` | int | Number of respondents |
| `Trump` | float | Trump support (%) |
| `Harris` | float | Harris support (%) |
| `Margin` | float | Trump minus Harris margin (percentage points) |

<!-- toc -->
## Contents
- [Setup](#setup)
- [1. Fetch and Parse](#1-fetch-and-parse)
- [2. Clean and Standardise](#2-clean-and-standardise)
- [3. Save](#3-save)
  - [wikipedia_polls.csv columns](#wikipedia_pollscsv-columns)


## Setup

In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')
import requests
from bs4 import BeautifulSoup
import pandas as pd

BRONZE_PATH = '../Data/1_Bronze/Polls/'
URL = (
    'https://en.wikipedia.org/wiki/'
    'Nationwide_opinion_polling_for_the_2024_United_States_presidential_election'
)

## 1. Fetch and Parse

In [2]:
# Step 1: Fetch and parse Wikipedia polling tables
headers = {'User-Agent': 'Mozilla/5.0 (compatible; SMWA-project/1.0)'}
response = requests.get(URL, headers=headers, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

all_tables = soup.find_all('table', class_='wikitable')

def col_text(table):
    return ' '.join(th.get_text(strip=True) for th in table.find_all('th')).lower()

poll_tables = [
    t for t in all_tables
    if 'trump' in col_text(t)
    and ('harris' in col_text(t) or 'kamala' in col_text(t))
    and len(t.find_all('tr')) > 10
]

raw_parts = []
for t in poll_tables:
    for d in pd.read_html(str(t)):
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = [
                ' '.join(str(x) for x in col if str(x) != 'nan').strip()
                for col in d.columns
            ]
        raw_parts.append(d)

df_raw = pd.concat(raw_parts, ignore_index=True)

## 2. Clean and Standardise

In [3]:

# Step 2: Standardise columns and clean values
rename = {}
for col in df_raw.columns:
    low = col.lower()
    if ('trump' in low or 'republican' in low) and 'harris' not in low and 'democrat' not in low:
        rename[col] = 'Trump'
    elif 'harris' in low or 'kamala' in low or 'democrat' in low:
        rename[col] = 'Harris'
    elif ('poll' in low or 'source' in low or 'firm' in low) and 'date' not in low:
        rename[col] = 'Pollster'
    elif 'date' in low or 'field' in low:
        rename[col] = 'Date'
    elif 'sample' in low or low.startswith('n '):
        rename[col] = 'SampleSize'

df = df_raw.rename(columns=rename).loc[:, lambda x: ~x.columns.duplicated(keep='first')]
keep = [c for c in ['Pollster', 'Date', 'SampleSize', 'Trump', 'Harris'] if c in df.columns]
df = df[keep]

for col in ['Trump', 'Harris']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
                   .str.replace('%', '', regex=False)
                   .str.replace('[^0-9.\-]', '', regex=True)
                   .pipe(pd.to_numeric, errors='coerce')
        )

def parse_date(s):
    s = re.sub(r'\[.*?\]', '', str(s)).strip()
    for sep in ['\u2013', '\u2014', '\u2012']:
        if sep in s:
            left, right = s.split(sep, 1)
            right = right.strip()
            if not re.search(r'[A-Za-z]', right.split(',')[0]):
                month = re.match(r'[A-Za-z]+', left.strip())
                if month:
                    right = month.group() + ' ' + right
            s = right
            break
    try:
        return pd.to_datetime(s.strip())
    except Exception:
        return pd.NaT

df['Date'] = df['Date'].apply(parse_date)

# Keep all 2024 polls — no July 5 cutoff here.
# The first available poll is July 6; keeping earlier polls allows the basetable
# to forward-fill July 5 with the last known poll value before our window starts.
df = (df[df['Date'].dt.year == 2024]
      .dropna(subset=['Trump', 'Harris'], how='all')
      .sort_values('Date')
      .reset_index(drop=True))
df['Margin'] = df['Trump'] - df['Harris']

## 3. Save

### wikipedia_polls.csv columns

| Column | Type | Description |
|---|---|---|
| `Pollster` | str | Name of the polling organisation |
| `Date` | date | Poll fieldwork end date |
| `SampleSize` | int | Number of respondents |
| `Trump` | float | Trump support (%) |
| `Harris` | float | Harris support (%) |
| `Margin` | float | Trump minus Harris margin (percentage points) |

In [4]:
# Step 3: Save to Bronze
os.makedirs(BRONZE_PATH, exist_ok=True)
out_path = os.path.join(BRONZE_PATH, 'wikipedia_polls.csv')
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows → {out_path}')
df.head()

Saved 256 rows → ../Data/1_Bronze/Polls/wikipedia_polls.csv


,Pollster,Date,SampleSize,Trump,Harris,Margin
0,McLaughlin & Associates[206],2024-06-24,"1,000 (LV)",47.0,42.0,5.0
1,Forbes/HarrisX[204],2024-06-30,"1,500 (RV)",53.0,47.0,6.0
2,CNN/SSRS[205],2024-06-30,"1,045 (RV)",47.0,45.0,2.0
3,Forbes/HarrisX[204],2024-06-30,"1,500 (RV)",43.0,38.0,5.0
4,Forbes/HarrisX[204],2024-06-30,"1,500 (RV)",43.0,40.0,3.0
